# Naive Regex Pruning

A simple keyword-based sentence filtering approach for context compression.

## How It Works

1. **Tokenize** the document into sentences
2. **Extract keywords** from the user's question (excluding common stop words)
3. **Score each sentence** by counting how many question keywords it contains
4. **Keep top sentences** that match at least one keyword (up to a configurable limit)

## Trade-offs

| Pros | Cons |
|------|------|
| Fast - no ML dependencies | May miss semantically relevant but lexically different sentences |
| Simple to understand and debug | Stop word list is language-specific |
| Deterministic output | Keyword matching is surface-level |

## Section A: Setup and Imports

In [ ]:
import re
from typing import List

## Section B: Core Helper Functions

In [ ]:
def sentence_split(text: str) -> List[str]:
    """Split text into sentences using regex."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def get_keywords(question: str) -> List[str]:
    """Extract meaningful keywords from question, excluding stop words."""
    stop_words = {
        "the", "for", "and", "this", "what", "with", "that", "from",
        "are", "was", "were", "have", "has", "had", "not", "but",
        "its", "all", "can", "two", "one", "each", "which", "their",
        "than", "they", "been", "some", "would", "could", "into",
        "about", "also", "how", "them", "these", "those", "will",
        "your", "our", "does"
    }
    keywords = [w.lower() for w in re.findall(r'[A-Za-z0-9]{3,}', question)]
    return [k for k in keywords if k not in stop_words]

## Section C: Naive Regex Pruning Method

In [ ]:
def naive_regex_prune(text: str, question: str, max_sentences: int = 30) -> str:
    """
    Prune context by keeping sentences that match question keywords.

    Args:
        text: The full document text
        question: The user's query
        max_sentences: Maximum sentences to keep

    Returns:
        Pruned text containing only relevant sentences
    """
    sentences = sentence_split(text)
    keywords = get_keywords(question)

    if not keywords:
        return ' '.join(sentences[:max_sentences])

    # Score sentences by keyword count
    scored = [(sum(1 for k in keywords if k in s.lower()), s) for s in sentences]
    scored.sort(key=lambda x: -x[0])

    # Keep sentences with at least one keyword match
    keep = [s for _, s in scored if _ > 0]

    if not keep:
        keep = sentences[:max_sentences]

    return ' '.join(keep[:max_sentences])

## Section D: Demo and Testing

In [ ]:
sample_text = (
    "The company reported strong Q3 earnings. Revenue grew by 15% year over year. "
    "The expansion of the Frankfurt server farm cost $14.73 million. This was partially "
    "offset by a $2.1 million green energy tax rebate. Employee headcount remained stable. "
    "Management raised full-year guidance citing strong demand in European markets."
)

sample_question = "What is the projected infrastructure capital expenditure for Q3 2026?"

print(f"Original length: {len(sample_text.split())} words")
print("\n" + "="*60)

result = naive_regex_prune(sample_text, sample_question)
print(f"\nPruned length: {len(result.split())} words")
print(f"\nPruned output:\n{result}")